In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- Step 5: Estimate BS locations (bring-your-own data)
# =============================================================================
# Step:         5 of 6
# Summary:      Estimate real base-station site coordinates from measured RSRP data -- anchors abstract servingCell ids to real-world locations for Step 6.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.0
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# Step 5 — Estimate BS locations (bring-your-own data)

Part of the **FUMD-AI training & postprocessing workflow** (step 5 of 6,
second postprocessing step).

**Purpose.** Unlike every other step in this workflow, this one does not
consume simulation output at all -- it works on **real measured RSRP
data** (cell id + GPS position + signal strength readings collected while
driving/walking a real coverage area) to estimate where each cell's
physical site actually is:

1. For each cell id, take the top-quantile (strongest) RSRP readings and
   use their median position as that cell's estimated location.
2. Cluster per-cell estimates that sit within a few meters of each other
   (DBSCAN, haversine metric) into physical sites, since one site commonly
   serves more than one cell id.
3. *(optional)* Plot the estimated sites over a real street network.

This is what lets Step 6 plot the model's predicted/actual migration
destinations on an actual map instead of the simulation's abstract
coordinate space -- see `bs_location.py`'s module docstring for why
matching a real site id to a simulation's `servingCell`/`destination` id is
a manual, domain-knowledge step this notebook does not attempt to automate.

**No bundled example.** This workflow has no real RSRP measurement data or
`.osm.pbf` street-network extract to ship, so `RSRP_CSV_PATH` (and
`OSM_PBF_PATH`, if you want the street-map plot) must point at your own
data -- the cell exits early with a clear message if `RSRP_CSV_PATH` isn't
set. Refactored from `estimacion_antenas.ipynb`'s working logic; its
exploratory cells (a pyrosm Helsinki demo, a folium custom-icon experiment)
were dropped, not carried over.

**No GPU needed.** This step is plain pandas/scikit-learn/matplotlib --
run it locally, not through `slurm/*.sbatch`.

**Outputs**, written under `OUTPUT_DIR`:
- `bs_per_cid.csv`, `bs_sites.csv`
- `coverage_and_sites.pdf`, and (if `OSM_PBF_PATH` is set) `sites_on_osm.pdf`


In [ ]:
import os
import sys

sys.path.insert(0, "src")
from fumd_training_workflow import bs_location as bl

In [ ]:
# ---- Parameters ----
# No default -- there is no bundled real-measurement dataset. Point this at
# your own RSRP measurement CSV (columns: lat, lon/long, cid, tech,
# rsrp_lte, rsrp_5g -- see bs_location.load_rsrp_measurements).
RSRP_CSV_PATH = None

TOP_QUANTILE = 0.20       # strongest fraction of each cell's readings used to estimate its location
CLUSTER_EPS_METERS = 1.0  # co-located cid estimates within this many meters are treated as one physical site

# Optional: a local .osm.pbf extract (e.g. from Geofabrik) for the
# street-map plot. Left unset by default -- the coverage-cloud plot above
# doesn't need it.
OSM_PBF_PATH = None
MAP_TITLE = "Base station locations"

OUTPUT_DIR = "bs_locations"

## 1. Load real RSRP measurements

In [ ]:
if not RSRP_CSV_PATH:
    raise ValueError(
        "RSRP_CSV_PATH is not set -- this step needs your own real RSRP "
        "measurement CSV (no example is bundled with this workflow). "
        "Set it in the parameters cell above, or via notebook_runner.run_step's "
        "`parameters` dict, then re-run."
    )

measurements = bl.load_rsrp_measurements(RSRP_CSV_PATH)
print(f"loaded {len(measurements)} measurement rows, {measurements['cid'].nunique()} distinct cell ids")

## 2. Estimate per-cell locations and cluster into sites

In [ ]:
bs_per_cid = bl.estimate_bs_per_cid(measurements, top_q=TOP_QUANTILE)
sites = bl.cluster_sites(bs_per_cid, eps_m=CLUSTER_EPS_METERS)

print(f"{len(bs_per_cid)} cell ids -> {len(sites)} physical site(s) after clustering (<= {CLUSTER_EPS_METERS} m)")
sites

## 3. Save tables and plots

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

bs_per_cid.to_csv(os.path.join(OUTPUT_DIR, "bs_per_cid.csv"), index=False)
sites.to_csv(os.path.join(OUTPUT_DIR, "bs_sites.csv"), index=False)

bl.plot_coverage_and_sites(measurements, sites, out_path=os.path.join(OUTPUT_DIR, "coverage_and_sites.pdf"))
print(f"saved bs_per_cid.csv, bs_sites.csv, coverage_and_sites.pdf -> {OUTPUT_DIR}")

if OSM_PBF_PATH:
    bl.plot_sites_on_osm(sites, OSM_PBF_PATH, out_path=os.path.join(OUTPUT_DIR, "sites_on_osm.pdf"), title=MAP_TITLE)
    print(f"saved sites_on_osm.pdf -> {OUTPUT_DIR}")
else:
    print("OSM_PBF_PATH not set -- skipping the street-map plot (coverage_and_sites.pdf above needs no OSM data)")